# Strands chatbot on ECS Fargate

Run the cells in order from the `strands-chatbot-fargate` folder (this notebook's folder).

1. Configure your settings
2. Deploy the **build** stack (ECR + CodeBuild)
3. Build the ARM64 image in CodeBuild (no local Docker needed)
4. Deploy the **service** stack (ECS cluster, ALB, Fargate service)
5. Chat with it
6. Tear down

In [ ]:
!pip install -q boto3 requests

## 1. Settings

In [ ]:
import boto3, json, os, shutil, time, zipfile, pathlib

region = boto3.Session().region_name or "us-east-1"
app_name = "strands-chatbot"
bucket = "my-existing-bucket"               # existing bucket in the same region

vpc_id = "vpc-0123456789abcdef0"
private_subnets = "subnet-aaa111,subnet-bbb222"   # tasks run here (need NAT or VPC endpoints)
alb_subnets = "subnet-aaa111,subnet-bbb222"       # private for internal ALB, public for internet-facing
alb_scheme = "internal"                           # or "internet-facing"
allowed_cidr = "10.0.0.0/8"                       # who can open the chat UI
model_id = ""                                     # Bedrock model / inference profile ID; blank = Strands default

build_stack = f"{app_name}-build"
service_stack = f"{app_name}-service"

cfn = boto3.client("cloudformation", region_name=region)
codebuild = boto3.client("codebuild", region_name=region)


def deploy(stack, template_path, params, capabilities=("CAPABILITY_IAM",)):
    """Create or update a stack and wait; returns its outputs."""
    body = open(template_path).read()
    p = [{"ParameterKey": k, "ParameterValue": v} for k, v in params.items()]
    try:
        cfn.create_stack(StackName=stack, TemplateBody=body, Parameters=p, Capabilities=list(capabilities))
        print(f"Creating {stack}...")
        cfn.get_waiter("stack_create_complete").wait(StackName=stack, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
    except cfn.exceptions.AlreadyExistsException:
        try:
            cfn.update_stack(StackName=stack, TemplateBody=body, Parameters=p, Capabilities=list(capabilities))
            print(f"Updating {stack}...")
            cfn.get_waiter("stack_update_complete").wait(StackName=stack, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
        except cfn.exceptions.ClientError as e:
            if "No updates are to be performed" not in str(e):
                raise
            print(f"{stack}: no changes")
    outs = cfn.describe_stacks(StackName=stack)["Stacks"][0].get("Outputs", [])
    return {o["OutputKey"]: o["OutputValue"] for o in outs}

## 2. Build stack (ECR repository + CodeBuild project)

In [ ]:
build_out = deploy(build_stack, "cfn/build.yaml", {"AppName": app_name, "SourceBucket": bucket})
build_out

## 3. Build and push the image
Zips the `app/` folder, uploads it, and runs CodeBuild. Re-run this cell (then step 4) whenever you change the app code.

In [ ]:
image_tag = str(int(time.time()))
source_key = f"{app_name}/source-{image_tag}.zip"

# Zip app/ without caches
if os.path.exists("source.zip"):
    os.remove("source.zip")
with zipfile.ZipFile("source.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in pathlib.Path("app").rglob("*"):
        if p.is_file() and "__pycache__" not in p.parts and p.suffix not in (".pyc", ".pyo"):
            z.write(p, p.relative_to("app"))
boto3.client("s3").upload_file("source.zip", bucket, source_key)

build = codebuild.start_build(
    projectName=build_out["BuildProjectName"],
    sourceLocationOverride=f"{bucket}/{source_key}",
    environmentVariablesOverride=[{"name": "IMAGE_TAG", "value": image_tag, "type": "PLAINTEXT"}],
)["build"]
print("Build started:", build["id"])

while True:
    b = codebuild.batch_get_builds(ids=[build["id"]])["builds"][0]
    print(b["currentPhase"], b["buildStatus"])
    if b["buildStatus"] != "IN_PROGRESS":
        break
    time.sleep(15)

if b["buildStatus"] != "SUCCEEDED":
    raise RuntimeError(f"Build failed - see logs: {b.get('logs', {}).get('deepLink')}")

image_uri = f"{build_out['RepositoryUri']}:{image_tag}"
print("Image:", image_uri)

## 4. Service stack (ECS cluster, ALB, Fargate service)
First deploy takes ~5-10 minutes. Later runs roll out the new image.

In [ ]:
service_out = deploy(service_stack, "cfn/service.yaml", {
    "AppName": app_name,
    "ImageUri": image_uri,
    "ModelId": model_id,
    "VpcId": vpc_id,
    "PrivateSubnetIds": private_subnets,
    "LoadBalancerSubnetIds": alb_subnets,
    "LoadBalancerScheme": alb_scheme,
    "AllowedCidr": allowed_cidr,
})
chat_url = service_out["ChatUrl"]
print("Open the chat UI at:", chat_url)

## 5. Test from the notebook
An **internal** ALB is only reachable from inside the VPC (or over VPN / Direct Connect). If this times out, open the URL from a machine on your network instead.

In [ ]:
import requests

r = requests.post(f"{chat_url}/api/chat",
                  json={"message": "What's the weather in Toronto right now?"}, timeout=120)
print(r.status_code)
data = r.json()
print(json.dumps(data, indent=2))

# Follow-up in the same conversation (the agent remembers context per session)
r = requests.post(f"{chat_url}/api/chat",
                  json={"message": "Should I bring a jacket?", "session_id": data.get("session_id")}, timeout=120)
print(r.json()["reply"])

### Logs (if something fails)

In [ ]:
logs = boto3.client("logs", region_name=region)
events = logs.filter_log_events(logGroupName=service_out["LogGroup"], limit=50)["events"]
for e in events[-50:]:
    print(e["message"])

## 6. Tear down
Deletes the service first (ALB, tasks, roles), then the build stack (ECR images are removed too).

In [ ]:
for stack in (service_stack, build_stack):
    cfn.delete_stack(StackName=stack)
    print(f"Deleting {stack}...")
    cfn.get_waiter("stack_delete_complete").wait(StackName=stack, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
print("Done")